# Répertorier les nouveaux fichiers d'événements à ingérer

Associez `lh_meridian_hr` comme lakehouse par défaut. Dans Fabric, marquez la cellule 2 comme cellule de paramètres.

Ce notebook pilote l'ingestion incrémentielle des extractions mensuelles `workforce_events_YYYY-MM.csv` :

1. Reçoit le `watermark` transmis par l'étape précédente du pipeline (`nb_setup_lakehouse`), qui crée et initialise `bronze.ingestion_watermark`. Ce notebook ne lit pas et ne crée pas la table de contrôle.
2. Répertorie les fichiers sources du dépôt GitHub via l'API Contents.
3. Ne conserve que les fichiers dont le mois est postérieur au watermark, les trie et limite leur nombre à `max_files_per_run`.
4. Renvoie une charge utile JSON consommée par le pipeline : `files` (chemins Copy relatifs pour le ForEach), `watermark` (mois à enregistrer après la réussite du lot) et `count`.

## Paramètres (marquer cette cellule comme cellule de paramètres)

**Résumé.** Le watermark entrant et la configuration de découverte des fichiers : emplacement des fichiers sources sur GitHub, préfixe utilisé pour construire les chemins Copy et limite de fichiers par exécution.

<details>
<summary>Détails ligne par ligne</summary>

- `watermark = "2020-12-01 00:00:00"` — **fourni par le pipeline** depuis la valeur de sortie du notebook de configuration. La valeur littérale sert uniquement de valeur de secours pour une exécution locale; elle n'est jamais lue dans la table de contrôle.
- `github_owner` / `github_repo` / `github_branch` / `github_path` — localisent le dossier source via l'API GitHub Contents.
- `copy_base_prefix = "events/"` — ajouté au nom de chaque fichier afin que le chemin renvoyé soit relatif à l'URL de base de l'activité Copy.
- `max_files_per_run = 60` — limite le nombre de fichiers renvoyés par une exécution.

</details>

In [ ]:
watermark = "2020-12-01 00:00:00"
github_owner = "modamin"
github_repo = "fabric-developer-training"
github_branch = "main"
github_path = "data/events"
copy_base_prefix = "events/"
max_files_per_run = 60

## Répertorier les fichiers plus récents que le watermark fourni

**Résumé.** Utilise le paramètre `watermark` tel quel, répertorie les fichiers sources via l'API GitHub Contents, ne conserve que les mois postérieurs à ce watermark (triés et limités), puis renvoie une charge utile JSON consommée par le pipeline.

<details>
<summary>Détails ligne par ligne</summary>

- Imports : `json`, `re`, `datetime` et `requests`. Aucun accès à Spark ou Delta n'est nécessaire; la table de contrôle est gérée en amont.
- `watermark_month = datetime.strptime(watermark, ...).strftime("%Y-%m")` — analyse le watermark fourni et le réduit à une chaîne `YYYY-MM` pour la comparaison.
- `requests.get(api_url, params={"ref": github_branch}, ...)` + `raise_for_status()` — appelle l'API GitHub Contents et échoue explicitement en cas de réponse d'erreur.
- La boucle `for entry in response.json()` conserve uniquement les vrais fichiers dont le nom correspond à `workforce_events_YYYY-MM.csv` et dont le mois est postérieur au mois du watermark; les correspondances sont collectées sous la forme `(month, path)`.
- `new_files.sort()` puis `new_files[:max_files_per_run]` — ordonne les fichiers chronologiquement et limite la taille du lot.
- `new_watermark` — le dernier mois sélectionné au format `YYYY-MM-01 00:00:00`, ou le watermark entrant inchangé lorsqu'il n'y a rien de nouveau.
- `notebookutils.notebook.exit(json.dumps(result))` — renvoie `files`, `watermark` et `count` au pipeline comme valeur de sortie.

</details>

In [ ]:
import json
import re
from datetime import datetime

import requests

watermark_month = datetime.strptime(watermark, "%Y-%m-%d %H:%M:%S").strftime("%Y-%m")

api_url = f"https://api.github.com/repos/{github_owner}/{github_repo}/contents/{github_path}"
response = requests.get(api_url, params={"ref": github_branch}, timeout=60)
response.raise_for_status()

name_pattern = re.compile(r"^workforce_events_(\d{4}-\d{2})\.csv$")
new_files = []
for entry in response.json():
    if entry.get("type") != "file":
        continue
    match = name_pattern.match(entry["name"])
    if not match:
        continue
    file_month = match.group(1)
    if file_month > watermark_month:
        new_files.append((file_month, copy_base_prefix + entry["name"]))

new_files.sort()
new_files = new_files[:max_files_per_run]

files = [path for _, path in new_files]
if new_files:
    new_watermark = new_files[-1][0] + "-01 00:00:00"
else:
    new_watermark = watermark

result = {
    "files": files,
    "watermark": new_watermark,
    "count": len(files),
}
print(result)
notebookutils.notebook.exit(json.dumps(result))